# Measure ice-sheet change with ICESat-2

**Terrain advanced · Python project · ICESat-2 ATL06**

Download real NASA **ATL06 Land Ice Height** granules for a repeating Greenland orbit track (RGT 277), extract segment heights with **h5py**, and compare two cycles to estimate elevation change.

---

## Learning objectives

- Access ICESat-2 data through **NASA Earthdata** (`earthaccess`)
- Navigate ATL06 **HDF5** groups (`gt1l/land_ice_segments`)
- Filter with ATL06 quality flags
- Compare repeat **Reference Ground Track (RGT)** passes for change

**Time:** ~3 hr · **Dataset:** [ICESat-2 explainer](../explainer.html?id=icesat-2)

---

## For mentors

- **Prerequisites:** Intermediate Python; comfort reading error messages.
- **Earthdata login:** Free at [urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov/). Paste credentials in **Step 0** (Colab) or use a local `.env` file.
- **Watch for:** Students skipping quality flags — only use segments with `atl06_quality_summary == 0`.
- **Runtime:** First run downloads ~200–400 MB of HDF5 (two granules); allow 5–10 min.

**Suggested flow:** Step 0 → imports → download → extract → plot → takeaway.

---

## Step 0 — NASA Earthdata login

**Colab:** Paste your Earthdata username and password below.

**Local:** Leave blank if `EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD` are in a `.env` file.

**Never commit credentials to GitHub.**

In [ ]:
EARTHDATA_USERNAME = ""  # paste here in Colab
EARTHDATA_PASSWORD = ""  # paste here in Colab

**Goal:** Install packages and log in to NASA Earthdata.

**You should see:** `Authentication successful` (or similar) from earthaccess.

In [ ]:
import os
from pathlib import Path

def pip_install(pkg):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["earthaccess", "h5py", "pandas", "numpy", "matplotlib"]:
    try:
        __import__(pkg)
    except ImportError:
        pip_install(pkg)

import earthaccess
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def load_earthdata_creds():
    user = EARTHDATA_USERNAME.strip() or os.environ.get("EARTHDATA_USERNAME", "").strip()
    pw = EARTHDATA_PASSWORD.strip() or os.environ.get("EARTHDATA_PASSWORD", "").strip()
    if not user or not pw:
        for env_path in [Path(".env"), Path("../.env")]:
            if not env_path.exists():
                continue
            for line in env_path.read_text().splitlines():
                if line.startswith("EARTHDATA_USERNAME="):
                    user = line.split("=", 1)[1].strip().strip('"').strip("'")
                if line.startswith("EARTHDATA_PASSWORD="):
                    pw = line.split("=", 1)[1].strip().strip('"').strip("'")
    return user, pw

user, pw = load_earthdata_creds()
if not user or not pw:
    raise ValueError(
        "Add NASA Earthdata credentials in Step 0 or .env. "
        "Register free: https://urs.earthdata.nasa.gov/"
    )

os.environ["EARTHDATA_USERNAME"] = user
os.environ["EARTHDATA_PASSWORD"] = pw
auth = earthaccess.login(strategy="environment")
print("Logged in to NASA Earthdata:", auth.authenticated if auth else "OK")

**Goal:** Download two ATL06 granules on the **same RGT** (277) in Greenland.

**Concept:** ICESat-2 repeats each RGT every ~91 days. We compare **cycle 03 (2019)** vs **cycle 14 (2022)** along granule `03` (central Greenland ice sheet).

**You should see:** Two `.h5` file paths downloaded.

**Mentor check-in:** *Why compare the same RGT instead of random orbits?*

In [ ]:
DATA_DIR = Path("atl06_granules")
DATA_DIR.mkdir(exist_ok=True)

GRANULE_PATTERNS = {
    "2019_cycle03": "*02770303*",
    "2022_cycle14": "*02771403*",
}
BEAM = "gt1l"  # strong left beam

downloaded = {}
for label, pattern in GRANULE_PATTERNS.items():
    results = earthaccess.search_data(short_name="ATL06", version="007", granule_name=pattern)
    if not results:
        raise RuntimeError(f"No granule found for pattern {pattern}")
    native = results[0]["meta"]["native-id"]
    print(f"Downloading {native} ({label})...")
    files = earthaccess.download(results[:1], local_path=str(DATA_DIR))
    downloaded[label] = Path(files[0])

downloaded

**Goal:** Read ATL06 HDF5 and extract land-ice segment heights.

**Concept:** ATL06 stores along-track segments in `gt1l/land_ice_segments` with height `h_li` and quality flag `atl06_quality_summary` (0 = good).

**You should see:** Thousands of valid segments per cycle after filtering.

In [ ]:
def extract_atl06_segments(h5path, beam=BEAM, lat_min=68, lat_max=71, lon_min=-52, lon_max=-45):
    with h5py.File(h5path, "r") as f:
        g = f[f"{beam}/land_ice_segments"]
        lat = g["latitude"][:]
        lon = g["longitude"][:]
        h_li = g["h_li"][:]
        x_atc = g["ground_track/x_atc"][:]
        sigma = g["h_li_sigma"][:]
        q = g["atl06_quality_summary"][:]
    good = (
        (q == 0)
        & np.isfinite(h_li)
        & (lat >= lat_min) & (lat <= lat_max)
        & (lon >= lon_min) & (lon <= lon_max)
    )
    return pd.DataFrame({
        "latitude": lat[good],
        "longitude": lon[good],
        "x_atc_m": x_atc[good],
        "h_li_m": h_li[good],
        "h_li_sigma_m": sigma[good],
    })

segments = {}
for label, path in downloaded.items():
    df = extract_atl06_segments(path)
    df["cycle"] = label
    segments[label] = df
    print(f"{label}: {len(df):,} valid segments")

all_seg = pd.concat(segments.values(), ignore_index=True)
all_seg.head()

**Goal:** Bin along-track distance and compare mean height between cycles.

**You should see:** A profile plot and a median elevation change (often negative = thinning).

**Mentor check-in:** *Why bin by `x_atc` instead of latitude alone?*

In [ ]:
BIN_M = 5000

def bin_profile(df):
    d = df.copy()
    d["x_bin"] = (d["x_atc_m"] // BIN_M) * BIN_M
    return (
        d.groupby("x_bin", as_index=False)
        .agg(
            h_mean_m=("h_li_m", "mean"),
            h_std_m=("h_li_m", "std"),
            n=("h_li_m", "size"),
        )
        .query("n >= 20")
    )

profiles = {label: bin_profile(df) for label, df in segments.items()}
early = profiles["2019_cycle03"].rename(columns={"h_mean_m": "h_early", "h_std_m": "h_early_std"})
late = profiles["2022_cycle14"].rename(columns={"h_mean_m": "h_late", "h_std_m": "h_late_std"})

change = early.merge(late, on="x_bin", how="inner")
change["dh_m"] = change["h_late"] - change["h_early"]
change["dh_sigma_m"] = np.sqrt(change["h_early_std"] ** 2 + change["h_late_std"] ** 2)

print(f"Median elevation change: {change['dh_m'].median():+.2f} m")
change.head()

**Goal:** Plot elevation profiles and change with uncertainty bands.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax0 = axes[0]
for label, prof in profiles.items():
    ax0.plot(prof["x_bin"] / 1000, prof["h_mean_m"], marker="o", ms=3, label=label)
ax0.set_ylabel("Land ice height (m)")
ax0.set_title("ATL06 land ice height · RGT 277 · gt1l · central Greenland")
ax0.legend()
ax0.grid(alpha=0.3)

ax1 = axes[1]
ax1.axhline(0, color="#4A5A54", lw=1)
ax1.errorbar(
    change["x_bin"] / 1000,
    change["dh_m"],
    yerr=change["dh_sigma_m"],
    fmt="o-",
    color="#C45C5C",
    ecolor="#D98E5A",
    capsize=2,
)
ax1.set_xlabel("Along-track distance (km)")
ax1.set_ylabel("Elevation change 2022 − 2019 (m)")
ax1.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## Final takeaway

**Student writes; mentor reviews:**

1. What is the **median elevation change** you computed?
2. Why do we filter `atl06_quality_summary == 0`?
3. What are two limits of comparing only two cycles?

**Methods sentence template:** *We compared ATL06 v007 granules 02770303 and 02771403 on RGT 277 (beam gt1l) over 68–71°N, using quality-flagged segments binned every 5 km along-track.*

**Next:** [ICESat-2 explainer](../explainer.html?id=icesat-2) · [NASA Earthdata](../explainer.html?id=nasa-earthdata)